# Generación y Visualización de Pares Siameses

Este notebook se usa **después** de extraer frames y recortar rostros con `01_dataset_preparation.ipynb`.

Una red Siamesa no aprende a clasificar personas por nombre; aprende a medir **similitud** entre dos imágenes.
Para eso necesita pares etiquetados:
- **Par positivo** (label 1): dos imágenes de la **misma persona**.
- **Par negativo** (label 0): dos imágenes de **personas distintas**.

Los pares se almacenan en CSVs (`train_pairs.csv`, `val_pairs.csv`, `test_pairs.csv`) — las imágenes no se copian.

## Parámetros

In [ ]:
# --- Parámetros editables ---

# Activar para ejecutar el script de generación de pares.
# Por defecto False para no sobreescribir CSVs existentes accidentalmente.
RUN_BUILD_PAIRS = False

# Límite total aproximado de pares a generar. None = sin límite.
MAX_PAIRS = None

# Semilla para reproducibilidad
SEED = 42

# Si True, pasa --overwrite al script
OVERWRITE_PAIRS = True

# Activar para mostrar imágenes de pares de ejemplo.
# Mantener False antes de hacer commit.
SHOW_PAIR_SAMPLES = False

# Cuántos pares de ejemplo mostrar por sección
NUM_SAMPLE_PAIRS = 3

## Importaciones y configuración

In [ ]:
import subprocess
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# cv2 solo se usa en la sección de visualización opcional
try:
    import cv2
    CV2_AVAILABLE = True
except ImportError:
    CV2_AVAILABLE = False
    print("[aviso] OpenCV no disponible — la visualización de imágenes estará deshabilitada.")

# Añadir la raíz del proyecto al path si este notebook se abre desde notebooks/
sys.path.insert(0, str(Path.cwd().parent))

from src.config import (
    PROJECT_ROOT,
    PAIRS_DIR,
    PROCESSED_DATASET_DIR,
    SUPPORTED_FACE_VIEWS,
)

print(f"PROJECT_ROOT          : {PROJECT_ROOT}")
print(f"PAIRS_DIR             : {PAIRS_DIR}")
print(f"PROCESSED_DATASET_DIR : {PROCESSED_DATASET_DIR}")
print(f"SUPPORTED_FACE_VIEWS  : {SUPPORTED_FACE_VIEWS}")

## Función auxiliar

In [ ]:
def run_command(command: list[str]) -> None:
    """Ejecuta un comando de shell y muestra su salida."""
    print("Ejecutando:", " ".join(command))
    print("-" * 60)
    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        capture_output=False,  # mostrar salida en tiempo real
    )
    print("-" * 60)
    if result.returncode != 0:
        print(f"[error] El proceso terminó con código {result.returncode}.")
    else:
        print("[ok] Comando completado exitosamente.")


def resolve_project_path(relative_path: str) -> Path:
    """Convierte una ruta relativa al proyecto en ruta absoluta."""
    return PROJECT_ROOT / relative_path

## Pares positivos y negativos

El script `src.dataset.build_pairs` genera tres CSVs balanceados (train / val / test).

| Tipo | Label | Criterio |
|------|-------|----------|
| Positivo | 1 | `person_a == person_b` (distintas vistas permitidas) |
| Negativo | 0 | `person_a != person_b` |

La proporción aproximada es **1:1**. Los CSVs guardan rutas relativas; las imágenes no se duplican.

## Generación de pares

In [ ]:
# Construir el comando según los parámetros definidos arriba
cmd = [sys.executable, "-m", "src.dataset.build_pairs", "--seed", str(SEED)]

if MAX_PAIRS is not None:
    cmd += ["--max-pairs", str(MAX_PAIRS)]

if OVERWRITE_PAIRS:
    cmd += ["--overwrite"]

if RUN_BUILD_PAIRS:
    run_command(cmd)
else:
    print("[deshabilitado] RUN_BUILD_PAIRS = False. Comando que se ejecutaría:")
    print(" ".join(cmd))
    print("\nCambia RUN_BUILD_PAIRS = True para generar los pares.")

## Estado de los CSVs

In [ ]:
# Verificar existencia de los tres archivos de pares
csv_files = {
    "train": PAIRS_DIR / "train_pairs.csv",
    "val":   PAIRS_DIR / "val_pairs.csv",
    "test":  PAIRS_DIR / "test_pairs.csv",
}

all_exist = True
for split, path in csv_files.items():
    status = "encontrado" if path.exists() else "NO encontrado"
    print(f"  [{split:5s}]  {status:15s}  {path}")
    if not path.exists():
        all_exist = False

if not all_exist:
    print("\nEjecuta primero la generación de pares activando RUN_BUILD_PAIRS = True.")

## Carga de CSVs

In [ ]:
# Cargar cada CSV que exista; los ausentes quedan como None
dfs = {}
for split, path in csv_files.items():
    if path.exists():
        dfs[split] = pd.read_csv(path)
        print(f"  [{split:5s}]  {len(dfs[split]):>6,} filas cargadas")
    else:
        dfs[split] = None
        print(f"  [{split:5s}]  (no disponible)")

# Vista rápida del primer split disponible
first_available = next((df for df in dfs.values() if df is not None), None)
if first_available is not None:
    display(first_available.head(3))

## Resumen por split

In [ ]:
total = 0
for split, df in dfs.items():
    if df is not None:
        n = len(df)
        total += n
        print(f"  {split:5s}: {n:>6,} pares")
    else:
        print(f"  {split:5s}: (no disponible)")

print(f"  {'total':5s}: {total:>6,} pares")

## Distribución de etiquetas

In [ ]:
rows = []
for split, df in dfs.items():
    if df is None:
        continue
    pos = int((df["label"] == 1).sum())
    neg = int((df["label"] == 0).sum())
    total_split = pos + neg
    rows.append({
        "split": split,
        "positive_pairs": pos,
        "negative_pairs": neg,
        "total_pairs": total_split,
        "positive_ratio": round(pos / total_split, 3) if total_split else 0,
        "negative_ratio": round(neg / total_split, 3) if total_split else 0,
    })

if rows:
    dist_df = pd.DataFrame(rows)
    display(dist_df)
else:
    print("No hay datos cargados.")

## Gráfico: distribución de etiquetas por split

In [ ]:
if rows:
    fig, ax = plt.subplots(figsize=(7, 4))
    x = range(len(dist_df))
    width = 0.35

    ax.bar([i - width / 2 for i in x], dist_df["positive_pairs"], width, label="Positivos")
    ax.bar([i + width / 2 for i in x], dist_df["negative_pairs"], width, label="Negativos")

    ax.set_xticks(list(x))
    ax.set_xticklabels(dist_df["split"])
    ax.set_ylabel("Número de pares")
    ax.set_title("Pares positivos vs negativos por split")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No hay datos para graficar.")

## Análisis de combinaciones de vistas

El proyecto soporta vistas múltiples (`frontal`, `left`, `right`, `mixed`). Esta sección muestra qué combinaciones de vistas aparecen en los pares, lo que ayuda a entender la diversidad del entrenamiento multi-vista.

In [ ]:
# Combinar todos los splits disponibles para el análisis de vistas
available_dfs = [df for df in dfs.values() if df is not None]

if available_dfs:
    all_pairs = pd.concat(available_dfs, ignore_index=True)

    # Crear columna de combinación de vistas
    all_pairs["view_pair"] = all_pairs["view_a"] + "-" + all_pairs["view_b"]

    view_counts = (
        all_pairs["view_pair"]
        .value_counts()
        .reset_index()
        .rename(columns={"index": "view_pair", "count": "count"})
    )
    # Pandas >= 2.0 ya nombra bien las columnas
    view_counts.columns = ["view_pair", "count"]

    print(f"Combinaciones de vistas únicas encontradas: {len(view_counts)}\n")
    display(view_counts.head(15))
else:
    print("No hay datos cargados.")

## Gráfico: combinaciones de vistas más frecuentes

In [ ]:
if available_dfs:
    top_n = 10
    top_views = view_counts.head(top_n)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(range(len(top_views)), top_views["count"])
    ax.set_xticks(range(len(top_views)))
    ax.set_xticklabels(top_views["view_pair"], rotation=30, ha="right")
    ax.set_ylabel("Número de pares")
    ax.set_title(f"Top {top_n} combinaciones de vistas")
    plt.tight_layout()
    plt.show()
else:
    print("No hay datos para graficar.")

## Verificación de integridad de los pares

In [ ]:
issues_found = False

for split, df in dfs.items():
    if df is None:
        continue

    print(f"--- {split} ---")

    # Rutas de imagen faltantes
    missing_a = df["image_a"].apply(lambda p: not resolve_project_path(p).exists()).sum()
    missing_b = df["image_b"].apply(lambda p: not resolve_project_path(p).exists()).sum()
    if missing_a or missing_b:
        print(f"  [!] Rutas faltantes — image_a: {missing_a}, image_b: {missing_b}")
        issues_found = True
    else:
        print("  [ok] Todas las rutas de imagen existen.")

    # Etiquetas inesperadas
    invalid_labels = (~df["label"].isin([0, 1])).sum()
    if invalid_labels:
        print(f"  [!] Etiquetas inválidas (no 0/1): {invalid_labels}")
        issues_found = True
    else:
        print("  [ok] Todas las etiquetas son 0 o 1.")

    # Positivos con personas distintas
    bad_pos = ((df["label"] == 1) & (df["person_a"] != df["person_b"])).sum()
    if bad_pos:
        print(f"  [!] Pares positivos con personas distintas: {bad_pos}")
        issues_found = True
    else:
        print("  [ok] Todos los positivos tienen la misma persona.")

    # Negativos con la misma persona
    bad_neg = ((df["label"] == 0) & (df["person_a"] == df["person_b"])).sum()
    if bad_neg:
        print(f"  [!] Pares negativos con la misma persona: {bad_neg}")
        issues_found = True
    else:
        print("  [ok] Todos los negativos tienen personas distintas.")

    print()

if not issues_found:
    print("Integridad verificada: no se encontraron problemas.")

## Visualización de pares de ejemplo

> **Aviso:** Esta sección puede mostrar rostros reales.
> **No guardar ni subir el notebook con outputs visibles.**
> Mantener `SHOW_PAIR_SAMPLES = False` antes de hacer commit.

### Pares positivos (label = 1)

In [ ]:
def show_pairs(df: pd.DataFrame, label: int, n: int, title_prefix: str) -> None:
    """Muestra n pares de imágenes lado a lado."""
    if df is None:
        print("No hay datos cargados.")
        return
    if not CV2_AVAILABLE:
        print("[aviso] OpenCV no disponible.")
        return

    subset = df[df["label"] == label].sample(min(n, len(df[df["label"] == label])), random_state=SEED)

    for _, row in subset.iterrows():
        path_a = resolve_project_path(row["image_a"])
        path_b = resolve_project_path(row["image_b"])
        if not path_a.exists() or not path_b.exists():
            print(f"  [omitido] Ruta no encontrada: {path_a.name} / {path_b.name}")
            continue

        img_a = cv2.cvtColor(cv2.imread(str(path_a)), cv2.COLOR_BGR2RGB)
        img_b = cv2.cvtColor(cv2.imread(str(path_b)), cv2.COLOR_BGR2RGB)

        fig, axes = plt.subplots(1, 2, figsize=(5, 2.8))
        axes[0].imshow(img_a)
        axes[0].set_title(f"{row['person_a']} / {row['view_a']}", fontsize=8)
        axes[0].axis("off")
        axes[1].imshow(img_b)
        axes[1].set_title(f"{row['person_b']} / {row['view_b']}", fontsize=8)
        axes[1].axis("off")
        fig.suptitle(f"{title_prefix} — label={label}", fontsize=9)
        plt.tight_layout()
        plt.show()


# Usar el split de entrenamiento para la vista previa
train_df = dfs.get("train")

if SHOW_PAIR_SAMPLES:
    show_pairs(train_df, label=1, n=NUM_SAMPLE_PAIRS, title_prefix="Par positivo")
else:
    print("Visualización deshabilitada (SHOW_PAIR_SAMPLES = False).")
    print("Actívala localmente para inspeccionar pares. No guardes los outputs.")

### Pares negativos (label = 0)

In [ ]:
if SHOW_PAIR_SAMPLES:
    show_pairs(train_df, label=0, n=NUM_SAMPLE_PAIRS, title_prefix="Par negativo")
else:
    print("Visualización deshabilitada (SHOW_PAIR_SAMPLES = False).")
    print("Actívala localmente para inspeccionar pares. No guardes los outputs.")

## Lista de verificación

Antes de continuar con el entrenamiento, confirma:

- [ ] CSVs generados en `data/pairs/`.
- [ ] Distribución positivo/negativo revisada (ratio ≈ 0.5).
- [ ] Combinaciones de vistas revisadas.
- [ ] Rutas de imagen verificadas sin errores.
- [ ] **Outputs del notebook limpios antes del commit** (`Kernel → Restart & Clear Output`).
- [ ] Siguiente paso: usar `src.dataset.dataloader` y entrenar la red Siamesa.